In [1]:
from dataclasses import dataclass
from typing import Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [2]:
import sys
import os

# Add the parent directory to sys.path to allow importing model_utils
current_dir = os.path.dirname(os.path.abspath("__file__"))
parent_dir = os.path.dirname(current_dir)
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

# Alternative if running interactively and __file__ is not defined:
# sys.path.append(os.path.abspath('..'))
sys.path.append('..')


In [3]:
def make_windows(
    series: np.ndarray,
    lookback: int,
    horizon: int,
) -> Tuple[np.ndarray, np.ndarray]:

    series = np.asarray(series, dtype=np.float32)
    if series.ndim == 1:
        series = series[:, None]  # (T, 1)

    T, C = series.shape
    N = T - lookback - horizon + 1
    if N <= 0:
        raise ValueError("Time series too short for given lookback/horizon.")

    X = np.zeros((N, lookback, C), dtype=np.float32)
    y = np.zeros((N, horizon, C), dtype=np.float32)

    for i in range(N):
        X[i] = series[i : i + lookback]
        y[i] = series[i + lookback : i + lookback + horizon]

    return X, y


In [4]:
class WindowDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.from_numpy(X)  # (N, L, C)
        self.y = torch.from_numpy(y)  # (N, H, C)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


In [5]:
class MLPForecaster(nn.Module):
    def __init__(
        self,
        lookback: int,
        in_channels: int,
        horizon: int,
        out_dim: int = 1,                 # often 1 for a single target variable
        hidden_sizes=(64, 32),
        dropout: float = 0.1,
        activation: str = "relu",
    ):
        super().__init__()
        self.lookback = lookback
        self.in_channels = in_channels
        self.horizon = horizon
        self.out_dim = out_dim

        act = {"relu": nn.ReLU, "gelu": nn.GELU, "tanh": nn.Tanh}[activation]

        layers = []
        prev = lookback * in_channels
        for hs in hidden_sizes:
            layers += [nn.Linear(prev, hs), act(), nn.Dropout(dropout)]
            prev = hs
        layers += [nn.Linear(prev, horizon * out_dim)]

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        # x: (B, L, C)
        b = x.size(0)
        x = x.reshape(b, -1)  # flatten - changed from .view to .reshape
        y = self.net(x)    # (B, H*out_dim)
        return y.view(b, self.horizon, self.out_dim)


In [6]:
# -----------------------------------------------------------------------------
# DATA LOADING & PREPROCESSING
# -----------------------------------------------------------------------------
DATA_PATH = '../dataset/subset_set.feather'  # Adjust path if needed
print(f"Loading data from {DATA_PATH}...")
# Use pd.read_feather instead of feather.read_table to get a DataFrame directly
df = pd.read_feather(DATA_PATH)

# Define constants
DATE_COL = 'date'
TARGET_COL = 'value'  # Assuming 'value' is the target variable (sales at the end of the day)


# Handle DATE_COL (ensure it is a column and not in the index)
if DATE_COL in df.index.names:
    if DATE_COL in df.columns:
        # If it's in both, drop the index version to avoid "cannot insert" error
        df = df.reset_index(drop=True)
    else:
        # If it's only in the index, move it to a column
        df = df.reset_index()

# If DATE_COL is NOT in columns (and wasn't in index), we have a problem, but assuming it exists somewhere.
# Just to be safe, if we still have a complex index, reset it.
if df.index.name == DATE_COL:
     df = df.reset_index(drop=True)
     
# Fallback: simple reset to ensure RangeIndex 0..N
df = df.reset_index(drop=True)

# Ensure date is datetime and sort
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values(DATE_COL)
df = df.reset_index(drop=True) # Final clean reset

# Prepare target variable
# Using 'value' as the target (sales at the end of the day)
y = df[TARGET_COL]

print(len(y))


# -----------------------------------------------------------------------------
# FEATURE ENGINEERING (EXOGENOUS VARIABLES)
# -----------------------------------------------------------------------------
# 1. Day of week (0=Monday, 6=Sunday)
df['day_of_week'] = df[DATE_COL].dt.dayofweek

# 2. Month (1=January, 12=December)
df['month'] = df[DATE_COL].dt.month

# 3. Day of month (1-31) - captures paydays
df['day_of_month'] = df[DATE_COL].dt.day

# 4. Is Weekend (Binary)
df['is_weekend'] = (df['day_of_week'] >= 5).astype(float)

# Christmas Day flag (1 on Dec 25, else 0)
df['is_christmas_day'] = ((df[DATE_COL].dt.month == 12) & (df[DATE_COL].dt.day == 25)).astype(float)

# 5. Promotions
# Identify all columns that start with 'promo_'
promo_cols = [col for col in df.columns if col.startswith('promo_')]
print(f"Found {len(promo_cols)} promotion columns: {promo_cols}")

# Ensure promo columns are numeric (float)
for col in promo_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(float)

# Define the list of exogenous features to use
#EXOG_COLS = None
EXOG_COLS = ['day_of_week', 'month', 'day_of_month', 'is_weekend', 'is_christmas_day'] + promo_cols




Loading data from ../dataset/subset_set.feather...
7610
Found 16 promotion columns: ['promo_type_FRPG', 'promo_value_FRPG', 'promo_type_GAS', 'promo_value_GAS', 'promo_type_BOGO', 'promo_value_BOGO', 'promo_type_DISC', 'promo_value_DISC', 'promo_type_CIRC', 'promo_value_CIRC', 'promo_type_CIRE', 'promo_value_CIRE', 'promo_type_CLCP', 'promo_value_CLCP', 'promo_type_LFPE', 'promo_value_LFPE']


In [7]:
@dataclass
class TrainConfig:
    lookback: int = 30
    horizon: int = 153
    batch_size: int = 32
    train_size: int = 455
    val_size : int = 153
    lr: float = 1e-3
    epochs: int = 30
    weight_decay: float = 1e-4
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# mlp_timeseries_forecasting.py
# A minimal MLP forecaster for time series using sliding windows (PyTorch)

from dataclasses import dataclass
from typing import Tuple, Optional, List

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

def train_mlp_forecaster(
    df: pd.DataFrame, 
    cfg: TrainConfig,
    target_channel: int = 0,    
    val_ratio: float = 0.2,
    hidden_sizes=(64, 32),
    target_col: str = TARGET_COL,
    exog_cols: Optional[List[str]] = EXOG_COLS,
    test_size: Optional[int] = None,
):
    """
    Trains an MLP or generic forecaster. 
    Handles multivariate input (Target + Exog) -> Multivariate Output.
    """
    
    cols = [target_col] + (exog_cols if exog_cols else [])
    data = df[cols].values
    
    test_start_idx = -test_size if test_size is not None else -cfg.horizon
    
    val_end_idx = test_start_idx 
    val_start_idx = val_end_idx - cfg.val_size
    train_end_idx = val_start_idx 
    
    train_data = data[:train_end_idx]
    
    # Scale ONLY Target Data (e.g. channel 0)
    scaler = RobustScaler()
    train_scaled = train_data.copy()
    train_target = train_data[:, target_channel:target_channel+1]
    train_scaled[:, target_channel:target_channel+1] = scaler.fit_transform(train_target)
    
    # Validation Data Context
    val_context_start_idx = val_start_idx - cfg.lookback
    val_context_data = data[val_context_start_idx : val_end_idx]
    
    val_scaled = val_context_data.copy()
    val_target = val_context_data[:, target_channel:target_channel+1]
    val_scaled[:, target_channel:target_channel+1] = scaler.transform(val_target)

    # Create windows
    X_train, y_train_full = make_windows(train_scaled, cfg.lookback, cfg.horizon)
    X_val, y_val_full = make_windows(val_scaled, cfg.lookback, cfg.horizon)

    # Target Selection
    y_train = y_train_full[:, :, target_channel : target_channel + 1]
    y_val = y_val_full[:, :, target_channel : target_channel + 1]

    C_in = X_train.shape[2]
    
    print(f"Train/Val Split Indices:")
    print(f"  Train End: {train_end_idx}")
    print(f"  Val Range: {val_start_idx} to {val_end_idx}")
    print(f"  Test Range: {test_start_idx} to End")
    print(f"Input Shape: (Batch, {cfg.lookback}, {C_in})")
    print(f"Output Shape: (Batch, {cfg.horizon}, 1)")

    train_loader = DataLoader(WindowDataset(X_train, y_train), batch_size=cfg.batch_size, shuffle=True)
    val_loader = DataLoader(WindowDataset(X_val, y_val), batch_size=cfg.batch_size, shuffle=False)
        
    model = MLPForecaster(
        lookback=cfg.lookback,
        in_channels=C_in,
        horizon=cfg.horizon,
        out_dim=1, 
        hidden_sizes=hidden_sizes,
        dropout=0.1,
        activation="relu",
    ).to(cfg.device)

    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    loss_fn = nn.MSELoss()

    best_val = float("inf")
    best_state = None
    
    train_losses = []
    val_losses = []

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        train_loss = 0.0
        for xb, yb in train_loader:
            xb = xb.to(cfg.device)
            yb = yb.to(cfg.device)

            pred = model(xb)
            loss = loss_fn(pred, yb)

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            train_loss += loss.item() * xb.size(0)
        train_loss /= len(train_loader.dataset)
        train_losses.append(train_loss)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(cfg.device)
                yb = yb.to(cfg.device)
                pred = model(xb)
                val_loss += loss_fn(pred, yb).item() * xb.size(0)
        val_loss /= len(val_loader.dataset)
        val_losses.append(val_loss)

        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    print(f"Best Val MSE: {best_val:.6f}")
    if best_state is not None:
        model.load_state_dict(best_state)

    return model, scaler, train_losses, val_losses

In [9]:
@torch.no_grad()
def forecast_next(
    model: MLPForecaster,
    scaler: RobustScaler,
    recent_history: np.ndarray,     # shape (lookback, C_in)
    target_channel: int = 0,
    device: Optional[str] = None,
) -> np.ndarray:
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    recent_history = np.asarray(recent_history, dtype=np.float32)
    if recent_history.ndim == 1:
        recent_history = recent_history[:, None]
        
    # Scale ONLY Target Data (channel 0)
    x = recent_history.copy()
    x[:, target_channel:target_channel+1] = scaler.transform(recent_history[:, target_channel:target_channel+1])
    
    x = torch.from_numpy(x).unsqueeze(0).to(device)  # (1, L, C)

    model = model.to(device).eval()
    
    pred_scaled = model(x).cpu().numpy()[0, :, 0]  # (H,)

    # Inverse transform strictly the target column
    pred = scaler.inverse_transform(pred_scaled.reshape(-1, 1)).flatten()
    return pred

@torch.no_grad()
def recursive_forecast(
    model: MLPForecaster,
    scaler: RobustScaler,
    recent_history: np.ndarray,  
    future_exog: np.ndarray,     
    target_channel: int = 0,
    device: Optional[str] = None,
) -> np.ndarray:
    """
    Recursively forecasts `horizon` steps using a 1-step-ahead model.
    Uses known `future_exog` for the next step's input.
    """
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    
    horizon = len(future_exog)
    
    # 1. Prepare initial input
    # recent_history: (L, C_in)
    recent_history = np.asarray(recent_history, dtype=np.float32)
    if recent_history.ndim == 1:
        recent_history = recent_history[:, None]
        
    # Scale ONLY the target channel of the initial history
    current_x_scaled = recent_history.copy()
    current_x_scaled[:, target_channel:target_channel+1] = scaler.transform(
        recent_history[:, target_channel:target_channel+1]
    )
    
    preds_scaled = []
    model = model.to(device).eval()
    
    # Window buffer (L, C_in) - starts with history
    input_window = current_x_scaled.copy()
    
    for i in range(horizon):
        # 1. Forecast next step
        x_tensor = torch.from_numpy(input_window).float().unsqueeze(0).to(device) # (1, L, C_in)
        y_pred = model(x_tensor) # (1, 1, 1) assuming horizon=1 model
        val_pred = y_pred.item()
        preds_scaled.append(val_pred)
        
        # 2. Update Window for next step
        # Shift everything back
        input_window = np.roll(input_window, -1, axis=0)
        
        # Insert the newly calculated prediction (scaled)
        input_window[-1, target_channel] = val_pred
        
        # Insert the raw UN-SCALED future exogenous variables directly
        # Assuming target_channel=0, exogs fall into [1:]
        input_window[-1, 1:] = future_exog[i]
        
    preds_scaled = np.array(preds_scaled).reshape(-1, 1)
    
    # Inverse transform predictions (target only)
    preds = scaler.inverse_transform(preds_scaled).flatten()
    return preds

In [10]:
from model_utils.plots import plot_results
import os
import matplotlib.pyplot as plt

seeds = [42, 123, 999]
FEATURE_COLS = [TARGET_COL] + EXOG_COLS

# Create base output directory for plots
base_output_dir = "product_forecasts"
os.makedirs(base_output_dir, exist_ok=True)

# 1. Identify all products
# Check column name again just in case
id_col = 'item_id' if 'item_id' in df.columns else 'id' 
product_ids = df[id_col].unique()
print(f"Found {len(product_ids)} products: {product_ids}")

for product_id in product_ids:
    print(f"\n{'#'*40}")
    print(f"Processing Product ID: {product_id}")
    print(f"{'#'*40}")
    
    # Filter data for this product
    product_df = df[df[id_col] == product_id].copy()
    product_df = product_df.sort_values(DATE_COL).reset_index(drop=True)
    
    # Check if we have enough data
    min_length = 300 
    if len(product_df) < min_length:
        print(f"Skipping product {product_id}: Not enough data (len={len(product_df)})")
        continue
        
    # Prepare data splits for plotting (Indices)
    # Based on TrainConfig: val_size=153, test_size=153
    test_size = 153
    val_size = 153
    
    # Slices
    test_slice = slice(-test_size, None)
    val_slice = slice(-(test_size + val_size), -test_size)
    train_slice = slice(None, -(test_size + val_size))
    
    # Dates
    train_index = product_df[DATE_COL].iloc[train_slice]
    val_index = product_df[DATE_COL].iloc[val_slice]
    test_index = product_df[DATE_COL].iloc[test_slice]
    
    # Values (Target)
    train_values = product_df[TARGET_COL].iloc[train_slice].values
    val_values = product_df[TARGET_COL].iloc[val_slice].values
    test_values = product_df[TARGET_COL].iloc[test_slice].values

    for seed in seeds:
        print(f"\n{'='*20} SEED {seed} (Product {product_id}) {'='*20}")
        
        # Create seed-specific directory
        seed_dir = os.path.join(base_output_dir, f"seed_{seed}")
        os.makedirs(seed_dir, exist_ok=True)

        # -------------------------------------------------------------------------
        # 1. Direct Multi-Step Forecasting (One-Shot)
        # -------------------------------------------------------------------------
        print(">> Training Direct Multi-Step Model (Horizon=153)...")
        cfg_direct = TrainConfig(
            lookback=48,
            horizon=153, 
            batch_size=32,
            train_size=455, 
            val_size=153,
            lr=1e-3,
            epochs=30,
            weight_decay=1e-4,
        )
        
        torch.manual_seed(seed)
        np.random.seed(seed)
        
        # Train
        model_direct, scaler_direct, train_loss_d, val_loss_d = train_mlp_forecaster(
            product_df, 
            cfg_direct, 
            target_channel=0, 
            target_col=TARGET_COL,
            exog_cols=EXOG_COLS,
            test_size=153
        )
        
        # Forecast Direct
        test_start_idx = -153
        history_start_idx = test_start_idx - cfg_direct.lookback
        
        # Get history for [Target, Exog...]
        recent_history_df = product_df[FEATURE_COLS].iloc[history_start_idx : test_start_idx]
        recent_history = recent_history_df.values
        
        pred_direct = forecast_next(
            model_direct, scaler_direct, recent_history, target_channel=0
        )

        # -------------------------------------------------------------------------
        # 2. Recursive Step-by-Step Forecasting (Iterative)
        # -------------------------------------------------------------------------
        print("\n>> Training Recursive Model (1-step ahead)...")
        cfg_recursive = TrainConfig(
            lookback=48,
            horizon=1,   # Train to predict 1 step
            batch_size=32,
            train_size=453,
            val_size=153, 
            lr=1e-3,
            epochs=30, 
            weight_decay=1e-4,
        )
        
        torch.manual_seed(seed)
        np.random.seed(seed)
        model_recursive, scaler_recursive, train_loss_r, val_loss_r = train_mlp_forecaster(
            product_df, 
            cfg_recursive, 
            target_channel=0,
            target_col=TARGET_COL,
            exog_cols=EXOG_COLS,
            test_size=153 
        )
        
        # Forecast Recursive
        # We need FUTURE Exogenous variables for the forecast horizon
        future_exog_df = product_df[EXOG_COLS].iloc[test_start_idx:]
        future_exog = future_exog_df.values
        
        if len(future_exog) != 153:
            future_exog = future_exog[:153]

        pred_recursive = recursive_forecast(
            model_recursive,
            scaler_recursive,
            recent_history, 
            future_exog,    
            target_channel=0
        )

        # -------------------------------------------------------------------------
        # Evaluation & Plotting
        # -------------------------------------------------------------------------
        
        # Direct Metrics
        rmse_direct = np.sqrt(mean_squared_error(test_values, pred_direct))
        mae_direct = mean_absolute_error(test_values, pred_direct)
        
        # Recursive Metrics
        rmse_recursive = np.sqrt(mean_squared_error(test_values, pred_recursive))
        mae_recursive = mean_absolute_error(test_values, pred_recursive)
        
        print(f"\nResults for Product {product_id} - Seed {seed}:")
        print(f"Direct:    RMSE={rmse_direct:.4f}, MAE={mae_direct:.4f}")
        print(f"Recursive: RMSE={rmse_recursive:.4f}, MAE={mae_recursive:.4f}")
        
        # Use plot_results for comprehensive plotting
        
        # Direct Plot
        plot_results(
            train_values, val_values, test_values, pred_direct,
            train_index, val_index, test_index,
            train_loss_d, val_loss_d,
            target_col=TARGET_COL,
            title=f'MLP Direct Forecast (Prod {product_id}, Seed {seed}) - RMSE: {rmse_direct:.2f}',
            save_path=os.path.join(seed_dir, f"direct_forecast_prod_{product_id}.png")
        )
        
        # Recursive Plot (Separate or Combined? plot_results handles separate)
        plot_results(
            train_values, val_values, test_values, pred_recursive,
            train_index, val_index, test_index,
            train_loss_r, val_loss_r,
            target_col=TARGET_COL,
            title=f'MLP Recursive Forecast (Prod {product_id}, Seed {seed}) - RMSE: {rmse_recursive:.2f}',
            save_path=os.path.join(seed_dir, f"recursive_forecast_prod_{product_id}.png")
        )
        
        # Also keep existing comparison plot of just the test period
        plt.figure(figsize=(12, 6))
        plt.plot(test_index, test_values, label='Actual', color='black', alpha=0.6, linewidth=2)
        plt.plot(test_index, pred_direct, label=f'Direct (RMSE={rmse_direct:.2f})', color='blue', linestyle='-')
        plt.plot(test_index, pred_recursive, label=f'Recursive (RMSE={rmse_recursive:.2f})', color='red', linestyle='--')
        plt.title(f'Comparison: Direct vs Recursive (Product {product_id} - Seed {seed})')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.savefig(os.path.join(seed_dir, f"comparison_prod_{product_id}.png"))
        plt.close()
        
        print(f"Saved plots to {seed_dir}")

Found 10 products: [ 26008 921558 213626 213625 213624 213628 213629 213630 213631 514230]

########################################
Processing Product ID: 26008
########################################

==================== SEED 42 (Product 26008) ====================
>> Training Direct Multi-Step Model (Horizon=153)...
Train/Val Split Indices:
  Train End: -306
  Val Range: -306 to -153
  Test Range: -153 to End
Input Shape: (Batch, 48, 22)
Output Shape: (Batch, 153, 1)
Best Val MSE: 0.207175

>> Training Recursive Model (1-step ahead)...
Train/Val Split Indices:
  Train End: -306
  Val Range: -306 to -153
  Test Range: -153 to End
Input Shape: (Batch, 48, 22)
Output Shape: (Batch, 1, 1)
Best Val MSE: 0.173055

Results for Product 26008 - Seed 42:
Direct:    RMSE=55.4114, MAE=33.5273
Recursive: RMSE=55.1716, MAE=30.7132
Saved plots to product_forecasts\seed_42

==================== SEED 123 (Product 26008) ====================
>> Training Direct Multi-Step Model (Horizon=153)...
Trai